# Continuous RUL Benchmark (CRB) — Google Colab (T4 GPU)

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all**

What happens: clone the repo → install deps → 2-minute smoke test → full benchmark
(all datasets × all models × 3 seeds) → leaderboard. Every finished run is saved immediately
(to Google Drive if mounted), so if Colab disconnects just **run all cells again — it resumes**.

## 0. Settings

In [ ]:
REPO_URL   = "https://github.com/Nafis878/Continuous-.git"
BRANCH     = "main"
WORKDIR    = "/content/Continuous"

USE_DRIVE  = True                                     # save results on Drive (needed to resume after disconnects)
DRIVE_ROOT = "/content/drive/MyDrive/CRB"             # results go to DRIVE_ROOT/results
NCMAPSS_DIR_ON_DRIVE = "/content/drive/MyDrive/N-CMAPSS"   # optional: N-CMAPSS_DS*.h5 files

SEEDS    = "42 43 44"
DATASETS = ""      # "" = all (FD001 FD002 FD003 FD004 PHM08 + N-CMAPSS if present), e.g. "FD001 FD004"
MODELS   = ""      # "" = all default models, e.g. "CATT+TCT CATT TCN Transformer"
RUN_SMOKE_TEST = True
RUN_SCANIA_STUDY = False   # separate classification study; downloads ~1.6 GB of LFS data

## 1. GPU check

In [ ]:
import subprocess, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "nvidia-smi not available")
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU (CPU works but is much slower).")

## 2. Clone / update the repo

In [ ]:
import os
# Private repo? Uncomment to use a token (never save the token in the notebook):
# from getpass import getpass
# REPO_URL = f"https://{getpass('GitHub token: ')}@github.com/Nafis878/Continuous-.git"

os.environ["GIT_LFS_SKIP_SMUDGE"] = "1"   # LFS data (SCANIA) is only fetched if RUN_SCANIA_STUDY
if os.path.isdir(os.path.join(WORKDIR, ".git")):
    !git -C {WORKDIR} fetch -q origin {BRANCH} && git -C {WORKDIR} reset -q --hard origin/{BRANCH}
else:
    !git clone -q --branch {BRANCH} {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git log --oneline -1
assert os.path.isfile("ml_pipeline/run_benchmark.py"), "clone failed - check REPO_URL / access"

## 3. Install dependencies

In [ ]:
!pip install -q -r ml_pipeline/requirements.txt
import numpy, pandas, sklearn, h5py
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| sklearn", sklearn.__version__, "| h5py", h5py.__version__)

## 4. Google Drive (results + optional N-CMAPSS data)

In [ ]:
OUT = f"{WORKDIR}/ml_pipeline/results/benchmark"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = f"{DRIVE_ROOT}/results"
    os.makedirs(OUT, exist_ok=True)
    if os.path.isdir(NCMAPSS_DIR_ON_DRIVE) and any(f.endswith(".h5") for f in os.listdir(NCMAPSS_DIR_ON_DRIVE)):
        os.environ["NCMAPSS_DIR"] = NCMAPSS_DIR_ON_DRIVE
        print("N-CMAPSS files:", sorted(f for f in os.listdir(NCMAPSS_DIR_ON_DRIVE) if f.endswith(".h5")))
    else:
        print("No N-CMAPSS .h5 on Drive -> N-CMAPSS datasets will be skipped.")
print("Results directory:", OUT)

## 5. Smoke test (~2 min): every model, 1 epoch, small subsets

In [ ]:
if RUN_SMOKE_TEST:
    !python ml_pipeline/run_benchmark.py --quick --fresh --out /content/crb_smoke
    import pandas as pd
    smoke = pd.read_csv("/content/crb_smoke/runs.csv")
    bad = smoke[smoke["status"] != "ok"]
    assert bad.empty, f"smoke test failures:\n{bad[['dataset','model','status']]}"
    print(f"Smoke test OK: {len(smoke)} runs")

## 6. Full benchmark

Long-running (hours for all models × datasets × 3 seeds). Progress is saved after every run;
if the runtime disconnects, re-run all cells and it continues where it stopped.

In [ ]:
args = f'--seeds {SEEDS} --out "{OUT}"'
if DATASETS.strip():
    args += f" --datasets {DATASETS}"
if MODELS.strip():
    args += f" --models {MODELS}"
print("python ml_pipeline/run_benchmark.py", args)
!python ml_pipeline/run_benchmark.py {args}

## 7. Leaderboard

In [ ]:
from IPython.display import Markdown, display
import pandas as pd
lb = os.path.join(OUT, "LEADERBOARD.md")
if os.path.exists(lb):
    display(Markdown(open(lb, encoding="utf-8").read()))
    display(pd.read_csv(os.path.join(OUT, "leaderboard_avg_rank.csv")))
else:
    print("No leaderboard yet.")

## 8. Download results (zip)

In [ ]:
import shutil, time
archive = shutil.make_archive(f"/content/crb-results-{time.strftime('%Y%m%d-%H%M%S')}", "zip", OUT)
print("zip:", archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print("download skipped:", exc)

## 9. Optional: SCANIA classification study + classic RUL scripts

Not part of CRB. Needs the SCANIA LFS data (~1.6 GB download).

In [ ]:
if RUN_SCANIA_STUDY:
    !git lfs install --skip-repo
    !git lfs pull --include "SCANIA-Component-X/data/*"
    !python ml_pipeline/run_family_comparison.py --seeds {SEEDS}
    !python ml_pipeline/run_scania_cost_tune.py --seeds {SEEDS}
    !python ml_pipeline/run_condition_rul.py --seeds {SEEDS}
    if USE_DRIVE:
        shutil.copytree("ml_pipeline/results/family_comparison", f"{DRIVE_ROOT}/family_comparison", dirs_exist_ok=True)